<a href="https://colab.research.google.com/github/sethkipsangmutuba/Data-Visualization/blob/main/Session_A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Structured Data: NumPy’s Structured Arrays

##Seth Kipsang

### Creating Structured Arrays

### More Advanced Compound Types

### RecordArrays: Structured Arrays with a Twist


##9.1 Structured Data: NumPy’s Structured Arrays

NumPy structured arrays store heterogeneous data using named fields within a single array, instead of separate arrays for each variable.

For example, data like name, age, and weight can be combined into one structured array using a compound `dtype` that defines field names and their types (e.g., strings, integers, floats).

Once created, values are assigned per field, and all records are stored together in one array structure.

Data can be accessed by field name, by index, or filtered using Boolean conditions on specific fields.

Although structured arrays are useful for simple heterogeneous data, more complex data handling is typically better done with Pandas DataFrames.

LOAD IRIS

In [219]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

iris_df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

iris_df['target'] = iris.target
iris_df['target']

,target
0,0
1,0
2,0
3,0
4,0
...,...
145,2
146,2
147,2
148,2


Extract heterogeneous components

In [220]:
names = np.array([f"sample_{i}" for i in range(len(iris_df))])
sepal_length = iris_df.iloc[:, 0].to_numpy()
sepal_width  = iris_df.iloc[:, 1].to_numpy()
petal_length = iris_df.iloc[:, 2].to_numpy()
petal_width  = iris_df.iloc[:, 3].to_numpy()
target       = iris_df['target'].to_numpy()

 Define structured dtype

In [221]:
dtype = np.dtype([
    ('name', 'U12'),
    ('sepal_length', 'f8'),
    ('sepal_width',  'f8'),
    ('petal_length', 'f8'),
    ('petal_width',  'f8'),
    ('target',       'i4')
])

Create structured array

In [222]:
data = np.empty(len(iris_df), dtype=dtype)

data['name'] = names
data['sepal_length'] = sepal_length
data['sepal_width']  = sepal_width
data['petal_length'] = petal_length
data['petal_width']  = petal_width
data['target']       = target


 Access

In [223]:
print("First record:\n", data[0])

print("\nSepal length (first 5):\n", data['sepal_length'][:5])

First record:
 ('sample_0', 5.1, 3.5, 1.4, 0.2, 0)

Sepal length (first 5):
 [5.1 4.9 4.7 4.6 5. ]


filtering

In [224]:
filtered = data[data['sepal_length'] > 5.5]
print("\nFiltered (first 5):\n", filtered[:5])



Filtered (first 5):
 [('sample_14', 5.8, 4. , 1.2, 0.2, 0) ('sample_15', 5.7, 4.4, 1.5, 0.4, 0)
 ('sample_18', 5.7, 3.8, 1.7, 0.3, 0) ('sample_50', 7. , 3.2, 4.7, 1.4, 1)
 ('sample_51', 6.4, 3.2, 4.5, 1.5, 1)]


 Diagnostics

In [225]:
print("\nDtype:", data.dtype)
print("Itemsize:", data.itemsize)
print("Total bytes:", data.nbytes)


Dtype: [('name', '<U12'), ('sepal_length', '<f8'), ('sepal_width', '<f8'), ('petal_length', '<f8'), ('petal_width', '<f8'), ('target', '<i4')]
Itemsize: 84
Total bytes: 12600


##9.2 Creating Structured Arrays

Structured array data types can be defined in multiple ways.

One approach is using a dictionary with field names and formats, where each field is assigned a specific data type.

Types can also be specified using standard Python types or NumPy dtypes for clarity.

Another method is defining a compound type as a list of tuples, each containing a field name and its type.

If field names are not needed, types can be given as a comma-separated string, resulting in default field names.

Type codes follow a pattern: optional endian indicator, type character, and size in bytes.

Common type codes represent bytes, integers, unsigned integers, floating points, complex numbers, strings, Unicode strings, and raw data.

LOAD + CLEAN IRIS (schema normalization)

In [226]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

iris_df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)
iris_df['target'] = iris.target

# Normalize column names → required for dtype alignment
iris_df.columns = [
    col.replace(" (cm)", "").replace(" ", "_")
    for col in iris_df.columns
]

N = len(iris_df)
N

150

 METHOD A: Dictionary-based dtype

In [227]:
dtype_dict = {
    'names': iris_df.columns.tolist(),
    'formats': ['f8', 'f8', 'f8', 'f8', 'i4']
}

data_dict = np.zeros(N, dtype=dtype_dict)

for col in iris_df.columns:
    data_dict[col] = iris_df[col]
dtype_dict

{'names': ['sepal_length',
  'sepal_width',
  'petal_length',
  'petal_width',
  'target'],
 'formats': ['f8', 'f8', 'f8', 'f8', 'i4']}

METHOD B: List of tuples (explicit schema)

In [228]:
dtype_tuples = np.dtype([
    ('sepal_length', np.float64),
    ('sepal_width',  np.float64),
    ('petal_length', np.float64),
    ('petal_width',  np.float64),
    ('target',       np.int32)
])

data_tuples = np.zeros(N, dtype=dtype_tuples)

for col in iris_df.columns:
    data_tuples[col] = iris_df[col]
data_tuples

array([(5.1, 3.5, 1.4, 0.2, 0), (4.9, 3. , 1.4, 0.2, 0),
       (4.7, 3.2, 1.3, 0.2, 0), (4.6, 3.1, 1.5, 0.2, 0),
       (5. , 3.6, 1.4, 0.2, 0), (5.4, 3.9, 1.7, 0.4, 0),
       (4.6, 3.4, 1.4, 0.3, 0), (5. , 3.4, 1.5, 0.2, 0),
       (4.4, 2.9, 1.4, 0.2, 0), (4.9, 3.1, 1.5, 0.1, 0),
       (5.4, 3.7, 1.5, 0.2, 0), (4.8, 3.4, 1.6, 0.2, 0),
       (4.8, 3. , 1.4, 0.1, 0), (4.3, 3. , 1.1, 0.1, 0),
       (5.8, 4. , 1.2, 0.2, 0), (5.7, 4.4, 1.5, 0.4, 0),
       (5.4, 3.9, 1.3, 0.4, 0), (5.1, 3.5, 1.4, 0.3, 0),
       (5.7, 3.8, 1.7, 0.3, 0), (5.1, 3.8, 1.5, 0.3, 0),
       (5.4, 3.4, 1.7, 0.2, 0), (5.1, 3.7, 1.5, 0.4, 0),
       (4.6, 3.6, 1. , 0.2, 0), (5.1, 3.3, 1.7, 0.5, 0),
       (4.8, 3.4, 1.9, 0.2, 0), (5. , 3. , 1.6, 0.2, 0),
       (5. , 3.4, 1.6, 0.4, 0), (5.2, 3.5, 1.5, 0.2, 0),
       (5.2, 3.4, 1.4, 0.2, 0), (4.7, 3.2, 1.6, 0.2, 0),
       (4.8, 3.1, 1.6, 0.2, 0), (5.4, 3.4, 1.5, 0.4, 0),
       (5.2, 4.1, 1.5, 0.1, 0), (5.5, 4.2, 1.4, 0.2, 0),
       (4.9, 3.1, 1.5, 0.2, 0),

METHOD C: Comma-separated string (no field names)

In [229]:
type_string = np.dtype('f8,f8,f8,f8,i4')

data_string = np.zeros(N, dtype=dtype_string)

# Default names: f0, f1, f2, ...
for i in range(4):
    data_string[f'f{i}'] = iris_df.iloc[:, i]

data_string['f4'] = iris_df['target']
data_string['f4']

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], dtype=int32)

METHOD D: Mixed Python + NumPy types

In [230]:
dtype_mixed = np.dtype([
    ('sepal_length', float),
    ('sepal_width',  float),
    ('petal_length', float),
    ('petal_width',  float),
    ('target',       int)
])

data_mixed = np.zeros(N, dtype=dtype_mixed)

for col in iris_df.columns:
    data_mixed[col] = iris_df[col]
data_mixed

array([(5.1, 3.5, 1.4, 0.2, 0), (4.9, 3. , 1.4, 0.2, 0),
       (4.7, 3.2, 1.3, 0.2, 0), (4.6, 3.1, 1.5, 0.2, 0),
       (5. , 3.6, 1.4, 0.2, 0), (5.4, 3.9, 1.7, 0.4, 0),
       (4.6, 3.4, 1.4, 0.3, 0), (5. , 3.4, 1.5, 0.2, 0),
       (4.4, 2.9, 1.4, 0.2, 0), (4.9, 3.1, 1.5, 0.1, 0),
       (5.4, 3.7, 1.5, 0.2, 0), (4.8, 3.4, 1.6, 0.2, 0),
       (4.8, 3. , 1.4, 0.1, 0), (4.3, 3. , 1.1, 0.1, 0),
       (5.8, 4. , 1.2, 0.2, 0), (5.7, 4.4, 1.5, 0.4, 0),
       (5.4, 3.9, 1.3, 0.4, 0), (5.1, 3.5, 1.4, 0.3, 0),
       (5.7, 3.8, 1.7, 0.3, 0), (5.1, 3.8, 1.5, 0.3, 0),
       (5.4, 3.4, 1.7, 0.2, 0), (5.1, 3.7, 1.5, 0.4, 0),
       (4.6, 3.6, 1. , 0.2, 0), (5.1, 3.3, 1.7, 0.5, 0),
       (4.8, 3.4, 1.9, 0.2, 0), (5. , 3. , 1.6, 0.2, 0),
       (5. , 3.4, 1.6, 0.4, 0), (5.2, 3.5, 1.5, 0.2, 0),
       (5.2, 3.4, 1.4, 0.2, 0), (4.7, 3.2, 1.6, 0.2, 0),
       (4.8, 3.1, 1.6, 0.2, 0), (5.4, 3.4, 1.5, 0.4, 0),
       (5.2, 4.1, 1.5, 0.1, 0), (5.5, 4.2, 1.4, 0.2, 0),
       (4.9, 3.1, 1.5, 0.2, 0),

INSPECT TYPE CODES + MEMORY STRUCTURE

In [231]:
print("Dtype (tuple method):\n", data_tuples.dtype)

Dtype (tuple method):
 [('sepal_length', '<f8'), ('sepal_width', '<f8'), ('petal_length', '<f8'), ('petal_width', '<f8'), ('target', '<i4')]


In [232]:
print("\nField offsets (bytes):")
for name in data_tuples.dtype.names:
    print(name, "->", data_tuples.dtype.fields[name][1])



Field offsets (bytes):
sepal_length -> 0
sepal_width -> 8
petal_length -> 16
petal_width -> 24
target -> 32


PREVIEW OUTPUTS

In [233]:
print("\nSample (dict method):\n", data_dict[:3])
print("\nSample (tuple method):\n", data_tuples[:3])
print("\nSample (string method):\n", data_string[:3])
print("\nSample (mixed method):\n", data_mixed[:3])


Sample (dict method):
 [(5.1, 3.5, 1.4, 0.2, 0) (4.9, 3. , 1.4, 0.2, 0) (4.7, 3.2, 1.3, 0.2, 0)]

Sample (tuple method):
 [(5.1, 3.5, 1.4, 0.2, 0) (4.9, 3. , 1.4, 0.2, 0) (4.7, 3.2, 1.3, 0.2, 0)]

Sample (string method):
 [(5.1, 3.5, 1.4, 0.2, 0) (4.9, 3. , 1.4, 0.2, 0) (4.7, 3.2, 1.3, 0.2, 0)]

Sample (mixed method):
 [(5.1, 3.5, 1.4, 0.2, 0) (4.9, 3. , 1.4, 0.2, 0) (4.7, 3.2, 1.3, 0.2, 0)]


##9.3 More Advanced Compound Types

Structured arrays can define fields that themselves contain arrays or matrices, allowing each element to store more complex data structures.

For example, a field can hold a fixed-size matrix, so each record includes both scalar values and multidimensional data.

Each element then behaves like a structured record with multiple components, including embedded arrays.

This approach is useful because the structure maps directly to C-style structs, enabling efficient memory access.

It is particularly valuable when interfacing NumPy with C or Fortran libraries that operate on structured data.

DEFINE ADVANCED COMPOUND DTYPE

In [234]:
#    - scalar field: target
#    - vector field: 4D feature vector
#    - matrix field: 2x2 reshaped feature block

dtype_advanced = np.dtype([
    ('features', 'f8', (4,)),   # 1D array field (vector)
    ('matrix',   'f8', (2, 2)), # 2D array field (matrix)
    ('target',   'i4')          # scalar field
])
dtype_advanced

dtype([('features', '<f8', (4,)), ('matrix', '<f8', (2, 2)), ('target', '<i4')])

CREATE STRUCTURED ARRAY

In [235]:
data = np.zeros(N, dtype=dtype_advanced)
data

array([([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0., 0., 0., 0.], [[0., 0.], [0., 0.]], 0),
       ([0.,

Populate vector field

In [236]:
data['features'] = iris_df.iloc[:, :4].to_numpy()
data['features']

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

Populate matrix field (reshape each row → 2x2)

In [237]:
data['matrix'] = iris_df.iloc[:, :4].to_numpy().reshape(N, 2, 2)
data['matrix']


array([[[5.1, 3.5],
        [1.4, 0.2]],

       [[4.9, 3. ],
        [1.4, 0.2]],

       [[4.7, 3.2],
        [1.3, 0.2]],

       [[4.6, 3.1],
        [1.5, 0.2]],

       [[5. , 3.6],
        [1.4, 0.2]],

       [[5.4, 3.9],
        [1.7, 0.4]],

       [[4.6, 3.4],
        [1.4, 0.3]],

       [[5. , 3.4],
        [1.5, 0.2]],

       [[4.4, 2.9],
        [1.4, 0.2]],

       [[4.9, 3.1],
        [1.5, 0.1]],

       [[5.4, 3.7],
        [1.5, 0.2]],

       [[4.8, 3.4],
        [1.6, 0.2]],

       [[4.8, 3. ],
        [1.4, 0.1]],

       [[4.3, 3. ],
        [1.1, 0.1]],

       [[5.8, 4. ],
        [1.2, 0.2]],

       [[5.7, 4.4],
        [1.5, 0.4]],

       [[5.4, 3.9],
        [1.3, 0.4]],

       [[5.1, 3.5],
        [1.4, 0.3]],

       [[5.7, 3.8],
        [1.7, 0.3]],

       [[5.1, 3.8],
        [1.5, 0.3]],

       [[5.4, 3.4],
        [1.7, 0.2]],

       [[5.1, 3.7],
        [1.5, 0.4]],

       [[4.6, 3.6],
        [1. , 0.2]],

       [[5.1, 3.3],
        [1.7, 

Populate scalar field

In [238]:
data['target'] = iris_df['target'].to_numpy()
data['target']

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], dtype=int32)

ACCESS PATTERNS

In [239]:
# Entire record
print("First record:\n", data[0])

First record:
 ([5.1, 3.5, 1.4, 0.2], [[5.1, 3.5], [1.4, 0.2]], 0)


In [240]:
# Vector field
print("\nFeature vector (first row):\n", data['features'][0])


Feature vector (first row):
 [5.1 3.5 1.4 0.2]


In [241]:
# Matrix field
print("\nMatrix (first row):\n", data['matrix'][0])


Matrix (first row):
 [[5.1 3.5]
 [1.4 0.2]]


In [242]:
# Single element inside matrix
print("\nMatrix[0,0] of first record:\n", data['matrix'][0][0, 0])


Matrix[0,0] of first record:
 5.1


VECTOR OPERATIONS ON SUBARRAY FIELD

In [243]:
# Compute row-wise norm of feature vectors
norms = np.linalg.norm(data['features'], axis=1)
print("\nFeature norms (first 5):\n", norms[:5])


Feature norms (first 5):
 [6.34507683 5.91692488 5.83609458 5.7497826  6.32139225]


FILTERING BASED ON EMBEDDED DATA

In [244]:
mask = data['features'][:, 0] > 5.5
filtered = data[mask]

print("\nFiltered records (first 3):\n", filtered[:3])


Filtered records (first 3):
 [([5.8, 4. , 1.2, 0.2], [[5.8, 4. ], [1.2, 0.2]], 0)
 ([5.7, 4.4, 1.5, 0.4], [[5.7, 4.4], [1.5, 0.4]], 0)
 ([5.7, 3.8, 1.7, 0.3], [[5.7, 3.8], [1.7, 0.3]], 0)]


MEMORY STRUCTURE INSPECTION

In [245]:
print("\nDtype:\n", data.dtype)
print("\nItemsize (bytes per record):", data.itemsize)
print("Total memory (bytes):", data.nbytes)


Dtype:
 [('features', '<f8', (4,)), ('matrix', '<f8', (2, 2)), ('target', '<i4')]

Itemsize (bytes per record): 68
Total memory (bytes): 10200


##9.4 RecordArrays: Structured Arrays with a Twist

Record arrays are similar to structured arrays but allow field access using attribute-style notation instead of dictionary-style indexing.

Fields can be accessed more conveniently (e.g., using dot notation), reducing typing effort.

However, this convenience comes with additional overhead, making access slower compared to standard structured arrays.

The choice between structured arrays and record arrays depends on whether ease of use or performance is more important.

 LOAD + CLEAN IRIS

In [246]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

iris_df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)
iris_df['target'] = iris.target

# Normalize column names
iris_df.columns = [
    col.replace(" (cm)", "").replace(" ", "_")
    for col in iris_df.columns
]

N = len(iris_df)
N

150

 DEFINE STRUCTURED DTYPE

In [247]:
dtype = np.dtype([
    ('sepal_length', 'f8'),
    ('sepal_width',  'f8'),
    ('petal_length', 'f8'),
    ('petal_width',  'f8'),
    ('target',       'i4')
])

CREATE STRUCTURED ARRAY

In [248]:
data_struct = np.zeros(N, dtype=dtype)

for col in iris_df.columns:
    data_struct[col] = iris_df[col]

CONVERT → RECORD ARRAY (the "twist")

In [249]:
data_rec = data_struct.view(np.recarray)
data_rec

rec.array([(5.1, 3.5, 1.4, 0.2, 0), (4.9, 3. , 1.4, 0.2, 0),
           (4.7, 3.2, 1.3, 0.2, 0), (4.6, 3.1, 1.5, 0.2, 0),
           (5. , 3.6, 1.4, 0.2, 0), (5.4, 3.9, 1.7, 0.4, 0),
           (4.6, 3.4, 1.4, 0.3, 0), (5. , 3.4, 1.5, 0.2, 0),
           (4.4, 2.9, 1.4, 0.2, 0), (4.9, 3.1, 1.5, 0.1, 0),
           (5.4, 3.7, 1.5, 0.2, 0), (4.8, 3.4, 1.6, 0.2, 0),
           (4.8, 3. , 1.4, 0.1, 0), (4.3, 3. , 1.1, 0.1, 0),
           (5.8, 4. , 1.2, 0.2, 0), (5.7, 4.4, 1.5, 0.4, 0),
           (5.4, 3.9, 1.3, 0.4, 0), (5.1, 3.5, 1.4, 0.3, 0),
           (5.7, 3.8, 1.7, 0.3, 0), (5.1, 3.8, 1.5, 0.3, 0),
           (5.4, 3.4, 1.7, 0.2, 0), (5.1, 3.7, 1.5, 0.4, 0),
           (4.6, 3.6, 1. , 0.2, 0), (5.1, 3.3, 1.7, 0.5, 0),
           (4.8, 3.4, 1.9, 0.2, 0), (5. , 3. , 1.6, 0.2, 0),
           (5. , 3.4, 1.6, 0.4, 0), (5.2, 3.5, 1.5, 0.2, 0),
           (5.2, 3.4, 1.4, 0.2, 0), (4.7, 3.2, 1.6, 0.2, 0),
           (4.8, 3.1, 1.6, 0.2, 0), (5.4, 3.4, 1.5, 0.4, 0),
           (5.2, 4.1, 1.

ACCESS COMPARISON

In [250]:
# Structured array (dictionary-style)
print("Structured access:\n", data_struct['sepal_length'][:5])

# Record array (attribute-style)
print("\nRecord array access:\n", data_rec.sepal_length[:5])

# Single record access
print("\nFirst record (recarray):\n", data_rec[0])

Structured access:
 [5.1 4.9 4.7 4.6 5. ]

Record array access:
 [5.1 4.9 4.7 4.6 5. ]

First record (recarray):
 (5.1, 3.5, 1.4, 0.2, 0)


FILTERING (same behavior)

In [251]:
filtered = data_rec[data_rec.sepal_length > 5.5]
print("\nFiltered (first 5):\n", filtered[:5])


Filtered (first 5):
 [(5.8, 4. , 1.2, 0.2, 0) (5.7, 4.4, 1.5, 0.4, 0) (5.7, 3.8, 1.7, 0.3, 0)
 (7. , 3.2, 4.7, 1.4, 1) (6.4, 3.2, 4.5, 1.5, 1)]


PERFORMANCE INSIGHT (micro comparison)

In [252]:
import time

# Structured access timing
start = time.time()
_ = data_struct['sepal_length'].sum()
t_struct = time.time() - start

# Record array access timing
start = time.time()
_ = data_rec.sepal_length.sum()
t_rec = time.time() - start

print("\nTiming comparison:")
print("Structured access time:", t_struct)
print("Record array access time:", t_rec)



Timing comparison:
Structured access time: 0.0002090930938720703
Record array access time: 0.00020837783813476562


MEMORY CHECK (identical)

In [253]:
print("\nSame memory object:",
      data_struct.base is data_rec.base or data_struct is data_rec)

print("Itemsize:", data_rec.itemsize)
print("Total bytes:", data_rec.nbytes)


Same memory object: False
Itemsize: 36
Total bytes: 5400
